# Big Data Analytics & Distributed Systems
## Assignment 6: Data Processing Engine Architecture & Pipeline Optimization using PySpark

### Learning Objectives
1. **Spark Architecture & Mechanics**: Deep dive into Driver Nodes, Cluster Managers, Executors, Jobs, Stages, Tasks, and execution flow.
2. **Lazy Evaluation & Lineage**: Understand how Spark builds a Directed Acyclic Graph (DAG) for physical query optimization.
3. **Schema Management**: Master explicit schema definition using `StructType` and `StructField` to optimize memory usage.
4. **Transformations & API Operations**: Apply Narrow and Wide transformations including column projections, filters, and shuffle aggregations.
5. **Optimization & Benchmark**: Analyze execution plans via `.explain(True)` and benchmark CSV vs. Parquet storage formats.

In [151]:

import os
import sys
import pyspark
from pyspark.sql import SparkSession
import time
import tempfile

# Verify system execution environment
print(f"Python Runtime Version  : {sys.version.split()[0]}")
print(f"PySpark Library Version : {pyspark.__version__}")

# Initialize active SparkSession with Hadoop Windows compatibility configurations
spark = SparkSession.builder \
    .appName("Superstore_Data_Engineering_Pipeline") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.driver.host", "localhost") \
    .config("spark.hadoop.fs.nativeio.impl.disable", "true") \
    .config("spark.hadoop.mapreduce.fileoutputcommitter.marksuccessfuljobs", "false") \
    .getOrCreate()

# Disable strict Hadoop file permission checks on local Windows filesystem
spark.sparkContext._jsc.hadoopConfiguration().set("fs.file.impl", "org.apache.hadoop.fs.LocalFileSystem")
spark.sparkContext._jsc.hadoopConfiguration().set("fs.nativeio.impl.disable", "true")

print("\nSparkSession successfully initialized!")
print(f"Application Name : {spark.sparkContext.appName}")
print(f"Spark Version    : {spark.version}")

Python Runtime Version  : 3.12.7
PySpark Library Version : 4.2.0

SparkSession successfully initialized!
Application Name : Superstore_Data_Engineering_Pipeline
Spark Version    : 4.2.0


### Step 1: Import PySpark SQL Modules & Data Types

In [152]:
# Import necessary PySpark SQL modules and explicit schema data types
from pyspark.sql import SparkSession # Core entry point for Spark SQL functionality
from pyspark.sql.types import (      # Structural types for defining custom schemas
    StructType, 
    StructField, 
    StringType, 
    IntegerType, 
    DoubleType, 
    DateType
)
from pyspark.sql.functions import (  # Built-in SQL functions for data transformations
    col,       # Function to reference DataFrame columns
    when,      # Evaluates conditional branch logic
    lit,       # Creates a literal/constant column value
    concat,    # Concatenates multiple string columns
    round,     # Rounds numerical values to specified decimal precision
    sum,       # Aggregate function: total sum
    avg,       # Aggregate function: arithmetic mean
    min,       # Aggregate function: minimum value
    max,       # Aggregate function: maximum value
    count,     # Aggregate function: row count
    countDistinct # Aggregate function: unique element count
)

# Display active Spark Session context details
print("SparkSession successfully initialized!")
print(f"Application Name: {spark.sparkContext.appName}")
print(f"Spark Version: {spark.version}")

# Import structural schema types
from pyspark.sql.types import (
    StructType, StructField, StringType, 
    IntegerType, DoubleType, DateType
)

# Import built-in SQL functions for transformation pipelines
from pyspark.sql.functions import (
    col, when, lit, concat, round, 
    sum, avg, min, max, count, countDistinct, to_date
)

print("All required PySpark functions successfully imported.")

SparkSession successfully initialized!
Application Name: Superstore_Data_Engineering_Pipeline
Spark Version: 4.2.0
All required PySpark functions successfully imported.


### Step 2: Read CSV Dataset Using Three Distinct Approaches

In [153]:
# Define target dataset path
csv_path = "Sample - Superstore.csv"

# Approach 1: Default Read (No headers, all columns read as String)
df_method_1 = spark.read.csv(csv_path)

# Approach 2: Header Enabled (First row as column names, all columns as String)
df_method_2 = spark.read.option("header", "true").csv(csv_path)

# Approach 3: Header Enabled + Schema Inference (Scans dataset twice to resolve types)
df_method_3 = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(csv_path)

print("Method 3 (Header + InferSchema) Inferred Schema:")
df_method_3.printSchema()

Method 3 (Header + InferSchema) Inferred Schema:
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



### Step 3: Display & Inspect Dataset Structure

In [154]:
# Load dataset using standard options for detailed inspection
df_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(csv_path)

# Display top 20 rows without truncating long text strings
print("Top 20 Dataset Sample Rows:")
df_raw.show(20, truncate=False) # Renders tabular format, truncate=False keeps full string length

# Print underlying DataFrame schema tree structure
print("\nDataFrame Schema Layout:")
df_raw.printSchema() # Outputs column hierarchy and inferred data types

# List all column header names in the DataFrame
print("\nDataFrame Column Names List:")
print(df_raw.columns) # Returns a standard Python list of column names

# List column names alongside their PySpark data types
print("\nDataFrame Column Names and Associated Data Types:")
print(df_raw.dtypes) # Returns list of tuples containing (column_name, data_type)

# Calculate total row count of the raw dataset
total_rows = df_raw.count() # Triggers an action to compute total record count across partitions
print(f"\nTotal Dataset Row Count: {total_rows}")


Top 20 Dataset Sample Rows:
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+----------------------------------------------------------------------------+--------+--------+--------+--------+
|Row ID|Order ID      |Order Date|Ship Date |Ship Mode     |Customer ID|Customer Name     |Segment    |Country      |City           |State         |Postal Code|Region |Product ID     |Category       |Sub-Category|Product Name                                                                |Sales   |Quantity|Discount|Profit  |
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+----------------------------------------------------------------------------+--------+--------+--------+-

### Step 4: Schema Handling & Custom StructType Schema Creation

In [155]:
# Construct an explicit production-grade custom schema using StructType and StructField
custom_schema = StructType([
    StructField("Row ID", IntegerType(), True),         # Unique row identifier integer key
    StructField("Order ID", StringType(), True),        # Unique order tracking code string
    StructField("Order Date", StringType(), True),      # Order date string (parsed later)
    StructField("Ship Date", StringType(), True),       # Shipment date string (parsed later)
    StructField("Ship Mode", StringType(), True),       # Logistics shipping class string
    StructField("Customer ID", StringType(), True),     # Customer identifier code string
    StructField("Customer Name", StringType(), True),   # Full name string of customer
    StructField("Segment", StringType(), True),         # Market segment classification string
    StructField("Country", StringType(), True),         # Country name location string
    StructField("City", StringType(), True),            # City location name string
    StructField("State", StringType(), True),           # State / Province name string
    StructField("Postal Code", StringType(), True),     # Postal zip code string (kept string to preserve zero prefixes)
    StructField("Region", StringType(), True),          # Geographic sales region string
    StructField("Product ID", StringType(), True),      # Unique inventory product code string
    StructField("Category", StringType(), True),        # Broad product category classification string
    StructField("Sub-Category", StringType(), True),    # Specific sub-category item string
    StructField("Product Name", StringType(), True),    # Full product item title string
    StructField("Sales", DoubleType(), True),           # Transaction sales total monetary value
    StructField("Quantity", IntegerType(), True),       # Quantity of units purchased integer
    StructField("Discount", DoubleType(), True),        # Percentage discount rate applied double
    StructField("Profit", DoubleType(), True)           # Net profit generated double
])

# Load raw CSV dataset using explicit custom schema
df_custom = spark.read \
    .option("header", "true") \
    .schema(custom_schema) \
    .csv(csv_path) # Reads dataset directly using pre-defined structure without scanning twice

# Print custom schema layout to confirm application
print("DataFrame Schema Applied via Custom StructType:")
df_custom.printSchema()

DataFrame Schema Applied via Custom StructType:
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



### Step 5: Data Exploration & Statistical Summary

In [156]:
# Generate standard descriptive statistics for numerical attributes
print("Descriptive Statistics for Sales, Quantity, Discount, and Profit:")
df_custom.select("Sales", "Quantity", "Discount", "Profit").describe().show() # Calculates count, mean, stddev, min, max

# Generate extended statistical summary including quartiles (25%, 50%, 75%)
print("Extended Summary Statistics with Percentile Breakdown:")
df_custom.select("Sales", "Profit").summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show()

# Retrieve distinct Product Categories in the dataset
print("Distinct Product Categories Available:")
df_custom.select("Category").distinct().show() # Returns distinct rows for Category attribute

# Compute distinct total counts for key business metrics
print("Distinct Counts of Order IDs and Customer IDs:")
df_custom.select(
    countDistinct("Order ID").alias("Distinct_Orders"),    # Counts unique active order IDs
    countDistinct("Customer ID").alias("Distinct_Customers") # Counts unique active customer IDs
).show()

Descriptive Statistics for Sales, Quantity, Discount, and Profit:
+-------+------------------+------------------+------------------+------------------+
|summary|             Sales|          Quantity|          Discount|            Profit|
+-------+------------------+------------------+------------------+------------------+
|  count|              9694|              9694|              9983|              9994|
|   mean|234.41818199917006|3.7909015886115123|0.3155949113492862|28.587912967780834|
| stddev| 631.7890112674363|2.2273345138146503| 3.314008629792499| 234.3891156047269|
|    min|             0.444|                 1|               0.0|         -6599.978|
|    max|          22638.48|                14|           295.056|          8399.976|
+-------+------------------+------------------+------------------+------------------+

Extended Summary Statistics with Percentile Breakdown:
+-------+------------------+------------------+
|summary|             Sales|            Profit|
+-------

### Step 6: Column Selection, Renaming, & Manipulation

In [157]:
# 1. Select specific business columns using select()
df_selected = df_custom.select("Order ID", "Customer Name", "Category", "Sales", "Profit") # Filters projection to required attributes
print("Selected Columns Sample:")
df_selected.show(5)

# 2. Rename columns using alias() within select()
df_aliased = df_custom.select(
    col("Order ID").alias("Transaction_ID"), # Renames 'Order ID' to 'Transaction_ID'
    col("Sales").alias("Gross_Revenue"),    # Renames 'Sales' to 'Gross_Revenue'
    col("Profit").alias("Net_Earnings")      # Renames 'Profit' to 'Net_Earnings'
)
print("Aliased Columns Sample:")
df_aliased.show(5)

# 3. Drop non-essential metadata columns using drop()
df_dropped = df_custom.drop("Row ID", "Postal Code", "Country") # Removes administrative metadata attributes
print("DataFrame Schema After Dropping Attributes:")
df_dropped.printSchema()

Selected Columns Sample:
+--------------+---------------+---------------+--------+--------+
|      Order ID|  Customer Name|       Category|   Sales|  Profit|
+--------------+---------------+---------------+--------+--------+
|CA-2016-152156|    Claire Gute|      Furniture|  261.96| 41.9136|
|CA-2016-152156|    Claire Gute|      Furniture|  731.94| 219.582|
|CA-2016-138688|Darrin Van Huff|Office Supplies|   14.62|  6.8714|
|US-2015-108966| Sean O'Donnell|      Furniture|957.5775|-383.031|
|US-2015-108966| Sean O'Donnell|Office Supplies|  22.368|  2.5164|
+--------------+---------------+---------------+--------+--------+
only showing top 5 rows
Aliased Columns Sample:
+--------------+-------------+------------+
|Transaction_ID|Gross_Revenue|Net_Earnings|
+--------------+-------------+------------+
|CA-2016-152156|       261.96|     41.9136|
|CA-2016-152156|       731.94|     219.582|
|CA-2016-138688|        14.62|      6.8714|
|US-2015-108966|     957.5775|    -383.031|
|US-2015-108966|

Step 7: Filtering Operations & Boolean Logic

In [167]:
# 1. Basic filtering using filter() and where() syntax
df_furniture = df_custom.filter(col("Category") == "Furniture") # Filters rows where Category equals 'Furniture'
df_west = df_custom.where(col("Region") == "West")              # Filters rows where Region equals 'West' using SQL synonym

# 2. Compound multi-condition filtering with AND (&) operator
df_high_val_tech = df_custom.filter(
    (col("Category") == "Technology") & (col("Sales") > 500.0) # Evaluates both conditions concurrently
)
print("High-Value Technology Sales (> $500):")
df_high_val_tech.select("Order ID", "Category", "Sales").show(5)

# 3. Compound multi-condition filtering with OR (|) operator
df_consumer_or_corporate = df_custom.filter(
    (col("Segment") == "Consumer") | (col("Segment") == "Corporate") # Matches rows satisfying either segment condition
)

# 4. Filtering using list membership with isin()
target_cities = ["Los Angeles", "Seattle", "San Francisco"] # List of priority geographic cities
df_target_cities = df_custom.filter(col("City").isin(target_cities)) # Filters matching cities
print("Sample Sales from Target Cities:")
df_target_cities.select("Order ID", "City", "Sales").show(5)

# 5. Pattern matching using like() SQL wildcards
df_office_binders = df_custom.filter(col("Sub-Category").like("%Binder%")) # Searches for substring pattern 'Binder'

# 6. Range filtering using between()
df_mid_range_sales = df_custom.filter(col("Sales").between(100.0, 300.0)) # Selects records with Sales in range [100, 300]
print("Mid-Range Sales Sample ($100 - $300):")
df_mid_range_sales.select("Order ID", "Sales").show(5)



High-Value Technology Sales (> $500):
+--------------+----------+--------+
|      Order ID|  Category|   Sales|
+--------------+----------+--------+
|CA-2014-115812|Technology| 907.152|
|CA-2014-115812|Technology| 911.424|
|CA-2016-117590|Technology|1097.544|
|CA-2016-105816|Technology| 1029.95|
|CA-2016-114104|Technology|  944.93|
+--------------+----------+--------+
only showing top 5 rows
Sample Sales from Target Cities:
+--------------+-----------+-------+
|      Order ID|       City|  Sales|
+--------------+-----------+-------+
|CA-2016-138688|Los Angeles|  14.62|
|CA-2014-115812|Los Angeles|  48.86|
|CA-2014-115812|Los Angeles|   7.28|
|CA-2014-115812|Los Angeles|907.152|
|CA-2014-115812|Los Angeles| 18.504|
+--------------+-----------+-------+
only showing top 5 rows
Mid-Range Sales Sample ($100 - $300):
+--------------+------+
|      Order ID| Sales|
+--------------+------+
|CA-2016-152156|261.96|
|CA-2014-115812| 114.9|
|CA-2014-143336|213.48|
|US-2015-150630| 124.2|
|CA-2016-

### Step 8: DataFrame Modifications, Transformations & Derived Columns

In [168]:
# Perform multi-column transformations using withColumn(), when(), concat(), and round()
df_transformed = df_custom \
    .withColumnRenamed("Sub-Category", "Sub_Category") \
    .withColumn("Sales", col("Sales").cast(DoubleType())) \
    .withColumn("Profit", col("Profit").cast(DoubleType())) \
    .withColumn(
        "Profit_Margin", 
        round((col("Profit") / col("Sales")) * 100, 2) # Calculates profit margin percentage and rounds to 2 decimals
    ) \
    .withColumn(
        "Performance_Tier",
        when(col("Profit") > 100.0, lit("High Profit")) \
        .when((col("Profit") >= 0.0) & (col("Profit") <= 100.0), lit("Low Profit")) \
        .otherwise(lit("Loss Making")) # Categorizes transactions into profit performance tiers
    ) \
    .withColumn(
        "Customer_Location",
        concat(col("City"), lit(", "), col("State")) # Combines City and State string values into a single location column
    )

# Display output sample of transformed columns
print("Modified DataFrame with Derived Metrics:")
df_transformed.select("Order ID", "Customer Name", "Customer_Location", "Sales", "Profit", "Profit_Margin", "Performance_Tier").show(10, truncate=False)

Modified DataFrame with Derived Metrics:
+--------------+---------------+------------------------+--------+--------+-------------+----------------+
|Order ID      |Customer Name  |Customer_Location       |Sales   |Profit  |Profit_Margin|Performance_Tier|
+--------------+---------------+------------------------+--------+--------+-------------+----------------+
|CA-2016-152156|Claire Gute    |Henderson, Kentucky     |261.96  |41.9136 |16.0         |Low Profit      |
|CA-2016-152156|Claire Gute    |Henderson, Kentucky     |731.94  |219.582 |30.0         |High Profit     |
|CA-2016-138688|Darrin Van Huff|Los Angeles, California |14.62   |6.8714  |47.0         |Low Profit      |
|US-2015-108966|Sean O'Donnell |Fort Lauderdale, Florida|957.5775|-383.031|-40.0        |Loss Making     |
|US-2015-108966|Sean O'Donnell |Fort Lauderdale, Florida|22.368  |2.5164  |11.25        |Low Profit      |
|CA-2014-115812|Brosina Hoffman|Los Angeles, California |48.86   |14.1694 |29.0         |Low Profit    

### Step 9: Null Value Detection, Cleansing, & Imputation Strategies

In [169]:
# 1. Audit missing / null record counts across target columns
print("Null Value Audit per Column:")
df_custom.select([
    count(when(col(c).isNull(), c)).alias(c) for c in ["Order ID", "Sales", "Postal Code", "Profit"] # Iterates and tallies missing counts
]).show()

# 2. Drop rows containing null values using dropna()
df_cleaned_any = df_custom.dropna(how="any") # Drops a row if ANY column value is null
df_cleaned_sales = df_custom.dropna(subset=["Sales", "Profit"]) # Drops a row ONLY if Sales or Profit is null

# 3. Fill missing values with default fallbacks using fillna()
df_filled = df_custom.fillna({
    "Postal Code": "00000", # Replaces missing zip codes with default string placeholder
    "Discount": 0.0,         # Replaces missing discount metrics with zero default double value
    "Profit": 0.0            # Replaces missing profit metrics with zero default double value
})

# 4. Replace specific values using replace()
df_standardized_segment = df_custom.replace("Home Office", "Corporate Office", subset=["Segment"]) # Standardizes value labels

print("Null Handling operations completed successfully.")

Null Value Audit per Column:
+--------+-----+-----------+------+
|Order ID|Sales|Postal Code|Profit|
+--------+-----+-----------+------+
|       0|  300|          0|     0|
+--------+-----+-----------+------+

Null Handling operations completed successfully.


### Step 10: Classification of PySpark Transformations

In [170]:
# Code Demonstration: Narrow vs Wide Transformations

# NARROW TRANSFORMATION: Pipelined locally inside individual memory partitions
df_narrow = df_custom.select("Order ID", "Category", "Sales") \
                     .filter(col("Sales") > 100.0) \
                     .withColumn("Sales_Tax", round(col("Sales") * 0.08, 2)) # Operates strictly in-place

# WIDE TRANSFORMATION: Triggers network shuffle and re-partitioning across cluster nodes
df_wide = df_custom.groupBy("Category") \
                   .agg(sum("Sales").alias("Total_Category_Sales")) \
                   .orderBy(col("Total_Category_Sales").desc()) # Re-organizes data globally

print("Narrow and Wide transformation pipelines executed lazily.")

Narrow and Wide transformation pipelines executed lazily.


### Step 11: Spark Actions & Driver Memory Safety

In [171]:
# 1. show(): Renders tabular output directly to stdout
print("Action 1: show(3)")
df_custom.show(3) # Fetches and formats top 3 records for visual inspection

# 2. count(): Calculates total row count across partitions
total_records = df_custom.count() # Returns total record count as a Python integer
print(f"Action 2: count() = {total_records}")

# 3. first(): Retrieves the first single Row object
first_row = df_custom.first() # Pulls the very first row from partition 0
print(f"Action 3: first() = {first_row['Order ID']} | {first_row['Customer Name']}")

# 4. take(n): Retrieves an array of the first n Row objects
take_three = df_custom.take(3) # Collects first 3 records safely to driver memory
print(f"Action 4: take(3) Count = {len(take_three)}")

# 5. head(n): Identical to take(n)
head_two = df_custom.head(2) # Collects top 2 records
print(f"Action 5: head(2) Count = {len(head_two)}")

# 6. collect(): Retrieves ALL rows from ALL cluster partitions to Driver Memory
# WARNING: Use cautiously on massive enterprise datasets!
sample_collected = df_custom.limit(5).collect() # Safe restricted collect demo
print(f"Action 6: collect() Limited Sample Count = {len(sample_collected)}")

Action 1: show(3)
+------+--------------+----------+----------+------------+-----------+---------------+---------+-------------+-----------+----------+-----------+------+---------------+---------------+------------+--------------------+------+--------+--------+-------+
|Row ID|      Order ID|Order Date| Ship Date|   Ship Mode|Customer ID|  Customer Name|  Segment|      Country|       City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name| Sales|Quantity|Discount| Profit|
+------+--------------+----------+----------+------------+-----------+---------------+---------+-------------+-----------+----------+-----------+------+---------------+---------------+------------+--------------------+------+--------+--------+-------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|Second Class|   CG-12520|    Claire Gute| Consumer|United States|  Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset Col...|261.96| 

### Step 12: Data Aggregation & Network Shuffle Mechanics

In [172]:
# Perform multi-metric group aggregation across product Category and Region
df_aggregated = df_custom.groupBy("Region", "Category").agg(
    round(sum("Sales"), 2).alias("Total_Sales"),       # Sums total revenue per group
    round(avg("Sales"), 2).alias("Average_Sales"),     # Computes mean transaction value
    min("Sales").alias("Min_Transaction_Sales"),       # Identifies minimum transaction value
    max("Sales").alias("Max_Transaction_Sales"),       # Identifies maximum transaction value
    count("Order ID").alias("Total_Orders_Count")      # Counts total orders fulfilled
).orderBy("Region", col("Total_Sales").desc())        # Sorts output by region and total revenue

print("Aggregated Sales Metrics by Region & Category:")
df_aggregated.show(12, truncate=False)

Aggregated Sales Metrics by Region & Category:
+-------+---------------+-----------+-------------+---------------------+---------------------+------------------+
|Region |Category       |Total_Sales|Average_Sales|Min_Transaction_Sales|Max_Transaction_Sales|Total_Orders_Count|
+-------+---------------+-----------+-------------+---------------------+---------------------+------------------+
|Central|Technology     |170401.53  |406.69       |1.98                 |17499.95             |420               |
|Central|Office Supplies|164616.2   |120.25       |0.444                |9892.74              |1422              |
|Central|Furniture      |162783.14  |344.88       |1.892                |3504.9               |481               |
|East   |Technology     |264872.08  |496.95       |2.97                 |11199.968            |535               |
|East   |Furniture      |205540.35  |351.95       |2.96                 |4416.174             |601               |
|East   |Office Supplies|201781.6

### Step 13: Core Performance Concepts & Physical Execution Plan Analysis

In [173]:
# Analyze physical query execution plan using explain(True)
print("================ Query Physical Execution Plan Breakdown ================")
df_custom.filter(col("Category") == "Technology") \
         .select("Order ID", "Category", "Sales") \
         .groupBy("Category") \
         .agg(sum("Sales")) \
         .explain(True) # Outputs Parsed Logical Plan, Analyzed Logical Plan, Optimized Plan, and Physical Plan

================ Query Physical Execution Plan Breakdown ================
== Parsed Logical Plan ==
'Aggregate ['Category], ['Category, unresolvedalias('sum('Sales))]
+- Project [Order ID#10704, Category#10717, Sales#10720]
   +- Filter (Category#10717 = Technology)
      +- Relation [Row ID#10703,Order ID#10704,Order Date#10705,Ship Date#10706,Ship Mode#10707,Customer ID#10708,Customer Name#10709,Segment#10710,Country#10711,City#10712,State#10713,Postal Code#10714,Region#10715,Product ID#10716,Category#10717,Sub-Category#10718,Product Name#10719,Sales#10720,Quantity#10721,Discount#10722,Profit#10723] csv

== Analyzed Logical Plan ==
Category: string, sum(Sales): double
Aggregate [Category#10717], [Category#10717, sum(Sales#10720) AS sum(Sales)#12160]
+- Project [Order ID#10704, Category#10717, Sales#10720]
   +- Filter (Category#10717 = Technology)
      +- Relation [Row ID#10703,Order ID#10704,Order Date#10705,Ship Date#10706,Ship Mode#10707,Customer ID#10708,Customer Name#10709,Segm

### Column Operations

Cell Description: Demonstrates narrow column projections using select(), column aliasing, and dropping unnecessary administrative attributes.

In [174]:
# Column selection
df_selected = df_custom.select("Order ID", "Customer Name", "Category", "Sales", "Profit")

# Column aliasing
df_aliased = df_custom.select(
    col("Order ID").alias("Transaction_ID"),
    col("Sales").alias("Gross_Revenue"),
    col("Profit").alias("Net_Earnings")
)

# Dropping unnecessary metadata columns
df_dropped = df_custom.drop("Row ID", "Postal Code", "Country")

print("Aliased Schema Sample:")
df_aliased.show(5)

Aliased Schema Sample:
+--------------+-------------+------------+
|Transaction_ID|Gross_Revenue|Net_Earnings|
+--------------+-------------+------------+
|CA-2016-152156|       261.96|     41.9136|
|CA-2016-152156|       731.94|     219.582|
|CA-2016-138688|        14.62|      6.8714|
|US-2015-108966|     957.5775|    -383.031|
|US-2015-108966|       22.368|      2.5164|
+--------------+-------------+------------+
only showing top 5 rows


In [175]:
# High-Value Technology filter (AND condition)
df_high_tech = df_custom.filter((col("Category") == "Technology") & (col("Sales") > 500.0))

# Target City membership filter (isin condition)
target_cities = ["Los Angeles", "Seattle", "San Francisco"]
df_target_cities = df_custom.filter(col("City").isin(target_cities))

# Mid-Range Sales filter (between condition)
df_mid_range = df_custom.filter(col("Sales").between(100.0, 300.0))

print("High-Value Technology Transactions Sample:")
df_high_tech.select("Order ID", "Category", "Sales").show(5)

High-Value Technology Transactions Sample:
+--------------+----------+--------+
|      Order ID|  Category|   Sales|
+--------------+----------+--------+
|CA-2014-115812|Technology| 907.152|
|CA-2014-115812|Technology| 911.424|
|CA-2016-117590|Technology|1097.544|
|CA-2016-105816|Technology| 1029.95|
|CA-2016-114104|Technology|  944.93|
+--------------+----------+--------+
only showing top 5 rows


In [176]:
# Apply transformations and compute derived metrics
df_transformed = df_custom \
    .withColumnRenamed("Sub-Category", "Sub_Category") \
    .withColumn("Order_Date_Parsed", to_date(col("Order Date"), "M/d/yyyy")) \
    .withColumn("Ship_Date_Parsed", to_date(col("Ship Date"), "M/d/yyyy")) \
    .withColumn("Profit_Margin", round((col("Profit") / col("Sales")) * 100, 2)) \
    .withColumn(
        "Performance_Tier",
        when(col("Profit") > 100.0, lit("High Profit"))
        .when((col("Profit") >= 0.0) & (col("Profit") <= 100.0), lit("Low Profit"))
        .otherwise(lit("Loss Making"))
    ) \
    .withColumn("Customer_Location", concat(col("City"), lit(", "), col("State")))

print("Transformed DataFrame Sample:")
df_transformed.select("Order ID", "Customer_Location", "Sales", "Profit_Margin", "Performance_Tier").show(5, truncate=False)

Transformed DataFrame Sample:
+--------------+------------------------+--------+-------------+----------------+
|Order ID      |Customer_Location       |Sales   |Profit_Margin|Performance_Tier|
+--------------+------------------------+--------+-------------+----------------+
|CA-2016-152156|Henderson, Kentucky     |261.96  |16.0         |Low Profit      |
|CA-2016-152156|Henderson, Kentucky     |731.94  |30.0         |High Profit     |
|CA-2016-138688|Los Angeles, California |14.62   |47.0         |Low Profit      |
|US-2015-108966|Fort Lauderdale, Florida|957.5775|-40.0        |Loss Making     |
|US-2015-108966|Fort Lauderdale, Florida|22.368  |11.25        |Low Profit      |
+--------------+------------------------+--------+-------------+----------------+
only showing top 5 rows


### Step 14: CSV vs. Parquet Format Benchmark & Comparison

In [177]:
from pyspark.sql import SparkSession

# Initialize SparkSession with Windows file system permission bypass flags
spark = SparkSession.builder \
    .appName("Superstore_Data_Engineering_Pipeline") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.driver.host", "localhost") \
    .config("spark.hadoop.fs.nativeio.impl.disable", "true") \
    .config("spark.hadoop.mapreduce.fileoutputcommitter.marksuccessfuljobs", "false") \
    .getOrCreate()

# Disable strict file permission checking in Hadoop's file system handler
spark.sparkContext._jsc.hadoopConfiguration().set("fs.file.impl", "org.apache.hadoop.fs.LocalFileSystem")
spark.sparkContext._jsc.hadoopConfiguration().set("fs.nativeio.impl.disable", "true")

print("SparkSession re-initialized with Windows Hadoop bypass flags!")

SparkSession re-initialized with Windows Hadoop bypass flags!


### Step 15: End-to-End Production Data Pipeline

In [178]:
# =========================================================================
# END-TO-END DATA ENGINEERING PIPELINE IMPLEMENTATION
# =========================================================================

# Step A: Ingest Raw Data using explicit production schema
raw_superstore_df = spark.read \
    .option("header", "true") \
    .schema(custom_schema) \
    .csv(csv_path) # Pipeline Ingestion Stage

# Step B: Cleanse missing records and filter out negative returns
cleaned_df = raw_superstore_df \
    .dropna(subset=["Order ID", "Customer ID", "Sales"]) \
    .filter(col("Sales") > 0.0) # Data Cleansing Stage

# Step C: Column selection and attribute filtering (Column Pruning / Filter Early)
projected_df = cleaned_df \
    .select("Order ID", "Order Date", "Segment", "Region", "Category", "Sub-Category", "Sales", "Quantity", "Profit") \
    .filter(col("Region").isin(["West", "East", "Central", "South"])) # Projection & Filtering Stage

# Step D: Apply transformations and calculate calculated business metrics
transformed_pipeline_df = projected_df \
    .withColumn("Profit_Margin", round((col("Profit") / col("Sales")) * 100, 2)) \
    .withColumn("High_Value_Flag", when(col("Sales") > 300.0, lit("YES")).otherwise(lit("NO"))) # Metric Enrichment Stage

# Step E: Aggregate metrics by Segment and Geographic Region
pipeline_summary_df = transformed_pipeline_df \
    .groupBy("Region", "Segment") \
    .agg(
        round(sum("Sales"), 2).alias("Total_Segment_Sales"), # Aggregates regional revenue total
        round(sum("Profit"), 2).alias("Total_Segment_Profit"), # Aggregates regional profit total
        count("Order ID").alias("Total_Orders_Processed")     # Tallies total order volume
    ) \
    .orderBy("Region", col("Total_Segment_Sales").desc()) # Output Aggregation Stage

# Step F: Display pipeline summary output
print("Final Data Pipeline Aggregation Results:")
pipeline_summary_df.show(20, truncate=False)

Final Data Pipeline Aggregation Results:
+-------+-----------+-------------------+--------------------+----------------------+
|Region |Segment    |Total_Segment_Sales|Total_Segment_Profit|Total_Orders_Processed|
+-------+-----------+-------------------+--------------------+----------------------+
|Central|Consumer   |249892.58          |9042.35             |1181                  |
|Central|Corporate  |157157.68          |18661.51            |652                   |
|Central|Home Office|90750.61           |12425.05            |427                   |
|East   |Consumer   |347820.22          |40418.12            |1421                  |
|East   |Corporate  |197498.14          |23642.71            |852                   |
|East   |Home Office|126875.7           |26611.18            |483                   |
|South  |Consumer   |194634.69          |26901.63            |821                   |
|South  |Corporate  |120417.63          |14565.9             |493                   |
|South  |Home

Summary Statistics & Distinct Counts

Cell Description: Computes central tendency metrics, standard deviations, quartiles, and distinct entity counts across core numerical attributes.

In [179]:
csv_path = "Sample - Superstore.csv"

# Re-create df_custom under the active SparkSession
df_custom = spark.read \
    .option("header", "true") \
    .schema(custom_schema) \
    .csv(csv_path)
# Basic summary statistics
print("Numerical Attributes Summary:")
df_custom.select("Sales", "Quantity", "Discount", "Profit").describe().show()

# Extended summary statistics with percentile breakdown
print("Sales & Profit Percentile Breakdown:")
# Now run your describe action
df_custom.select("Sales", "Quantity", "Discount", "Profit").describe().show()

# Unique business entity counts
print("Distinct Orders & Customers Count:")
df_custom.select(
    countDistinct("Order ID").alias("Distinct_Orders"),
    countDistinct("Customer ID").alias("Distinct_Customers")
).show()

Numerical Attributes Summary:
+-------+------------------+------------------+------------------+------------------+
|summary|             Sales|          Quantity|          Discount|            Profit|
+-------+------------------+------------------+------------------+------------------+
|  count|              9694|              9694|              9983|              9994|
|   mean|234.41818199917006|3.7909015886115123|0.3155949113492862|28.587912967780834|
| stddev| 631.7890112674363|2.2273345138146503| 3.314008629792499| 234.3891156047269|
|    min|             0.444|                 1|               0.0|         -6599.978|
|    max|          22638.48|                14|           295.056|          8399.976|
+-------+------------------+------------------+------------------+------------------+

Sales & Profit Percentile Breakdown:
+-------+------------------+------------------+------------------+------------------+
|summary|             Sales|          Quantity|          Discount|      

Spark Action Execution:
Triggers execution via Spark Actions (show, count, first, take, head, collect), incorporating memory safeguards to avoid driver OutOfMemory (OOM) errors.

In [180]:
print(f"Total Count Action : {df_custom.count()}")

first_row = df_custom.first()
print(f"First Row Action   : {first_row['Order ID']} | {first_row['Customer Name']}")

take_sample = df_custom.take(3)
print(f"Take(3) Action     : Fetched {len(take_sample)} rows safely.")

# Driver Memory Safeguard: Limit collect size to prevent Driver OOM crashes
collect_sample = df_custom.limit(5).collect()
print(f"Collect Action     : Safely retrieved {len(collect_sample)} rows to driver memory.")


Total Count Action : 9994
First Row Action   : CA-2016-152156 | Claire Gute
Take(3) Action     : Fetched 3 rows safely.
Collect Action     : Safely retrieved 5 rows to driver memory.


### Step 16: Output Persistence & Save Modes

In [181]:
from pyspark.sql import SparkSession

# Stop any active session to apply new configurations
if 'spark' in locals():
    spark.stop()

# Re-initialize SparkSession with Windows Hadoop permission overrides
spark = SparkSession.builder \
    .appName("Superstore_Data_Engineering_Pipeline") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.driver.host", "localhost") \
    .config("spark.hadoop.fs.nativeio.impl.disable", "true") \
    .config("spark.hadoop.mapreduce.fileoutputcommitter.marksuccessfuljobs", "false") \
    .getOrCreate()

# Explicitly set Hadoop local file system handler configuration
spark.sparkContext._jsc.hadoopConfiguration().set("fs.file.impl", "org.apache.hadoop.fs.LocalFileSystem")
spark.sparkContext._jsc.hadoopConfiguration().set("fs.nativeio.impl.disable", "true")

print("SparkSession re-initialized with Windows Hadoop bypass flags!")

SparkSession re-initialized with Windows Hadoop bypass flags!


Pipeline Implementation

 Executes an end-to-end data pipeline from raw ingestion, null filtering, transformation, metric calculation, to summary aggregation.

In [182]:
# Step A: Ingestion with custom schema
raw_df = spark.read.option("header", "true").schema(custom_schema).csv(csv_path)

# Step B: Cleansing & Filtering
cleaned_df = raw_df.dropna(subset=["Order ID", "Customer ID", "Sales"]).filter(col("Sales") > 0.0)

# Step C: Projection & Selection
projected_df = cleaned_df.select("Order ID", "Order Date", "Segment", "Region", "Category", "Sub-Category", "Sales", "Profit")

# Step D: Transformation & Metric Calculation
transformed_pipeline_df = projected_df \
    .withColumn("Profit_Margin", round((col("Profit") / col("Sales")) * 100, 2)) \
    .withColumn("High_Value_Flag", when(col("Sales") > 300.0, lit("YES")).otherwise(lit("NO")))

# Step E: Aggregation
pipeline_summary_df = transformed_pipeline_df.groupBy("Region", "Segment").agg(
    round(sum("Sales"), 2).alias("Total_Segment_Sales"),
    round(sum("Profit"), 2).alias("Total_Segment_Profit"),
    count("Order ID").alias("Total_Orders_Processed")
).orderBy("Region", col("Total_Segment_Sales").desc())

print("End-to-End Pipeline Output:")
pipeline_summary_df.show(20, truncate=False)

End-to-End Pipeline Output:
+-------+-----------+-------------------+--------------------+----------------------+
|Region |Segment    |Total_Segment_Sales|Total_Segment_Profit|Total_Orders_Processed|
+-------+-----------+-------------------+--------------------+----------------------+
|Central|Consumer   |249892.58          |9042.35             |1181                  |
|Central|Corporate  |157157.68          |18661.51            |652                   |
|Central|Home Office|90750.61           |12425.05            |427                   |
|East   |Consumer   |347820.22          |40418.12            |1421                  |
|East   |Corporate  |197498.14          |23642.71            |852                   |
|East   |Home Office|126875.7           |26611.18            |483                   |
|South  |Consumer   |194634.69          |26901.63            |821                   |
|South  |Corporate  |120417.63          |14565.9             |493                   |
|South  |Home Office|73931

In [ ]:
### Conclusion:
This assignment successfully demonstrated the architectural mechanics, evaluation model, and optimization techniques of Apache Spark using PySpark through a practical implementation on the Superstore dataset.
### Key Takeaways & Practical Learnings:
## Explicit Schema Enforcement: 
    Defining explicit schemas using StructType and StructField eliminates unnecessary job executions caused by inferSchema, improving ingestion efficiency and ensuring runtime data safety.
### Lazy Evaluation & Execution DAG: 
    Spark builds an optimized Directed Acyclic Graph (DAG) for operations. Transformations remain unexecuted until an explicit action (e.g., .show(), .count(), .collect()) is triggered. 
## Transformation Mechanics (Narrow vs. Wide):
Narrow Transformations (filter, select, withColumn) operate locally within individual partitions without data movement across nodes.  
Wide Transformations (groupBy, orderBy) require network shuffling across partitions to aggregate data globally.  
## Execution Plan Optimization: 
Inspecting physical execution plans via .explain(True) verifies logical optimization steps, pushdown filters, and partition hashing applied by Spark Catalyst Optimizer. 
## Driver Memory Safeguards: 
Actions returning data to the driver process (such as .collect()) are bounded using .limit() to prevent OutOfMemory (OOM) exceptions on enterprise-scale workloads. 
## Production Pipeline Design: 
    Modularizing data workflows—from schema ingestion, null cleansing, derived feature engineering, to summary aggregation—produces reproducible, scalable data pipelines suited for distributed environments.  